# Assignment 3

# Basketball Shot Data Exploration

After playing basketball from childhood through college, my career in sports and my interest in deeping my professional connection to basketball I figured it would be good to explore some shot data from the 2015-2020 NBA seasons. We'll be visualizing trends of shot type distributions, efficiency by distance, shot locations and yearly shifts in efficiency by team and player. The purpose of this exploration is to identify emerging trends and interact with the data through filters. We'll use scatter plots to track shot locations, stacked bar to view type distributions, histograms to view efficiency by distance

## Scatter Plot Visualization Technique

Scatter plots are primarily used to show relationships between two variables (numeric). The points on a scatter plot can aid one in determining trends within the data.
The points on scatter plots can be adjusted to change shape, color, and size depending on what it is you want to visualize. 

In this notebook, we'll use scatter plots to chart shot locations based on their X and Y components on a basketball court. The final scatter plot will also be interactive to allow for other
explorations into how shot location changes based on shot type, player, make or miss, and game situation.

___

## Considerations for Library Selection
We'll use Plotly for our exploration as interactivity is native out of the box, it has a seamless integration with Streamlit for displaying our dashboard and it's efficient in spatial rendering.

Streamlit is a light weight way to turn data scripts into ewb apps without having to write a bunch of web script, the layout utilities are easy to use and it can be deployed from our github repository to a live URL quickly.

## Installing the Libraries

run the following command

%pip install streamlit plotly pandas

In [39]:
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import base64
import os

## Step 1: Page Configuration

Set up the dashboard page and configurations in streamlit

In [3]:
st.set_page_config(page_title = "NBA Shot Analytics", layout='wide',initial_sidebar_state="expanded")
st.title("NBA Shot Analysis Dashboard (2015-2020)")
st.markdown("An interactive look into shooting patterns and player efficiency")

2026-07-15 19:34:56.688 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:34:56.689 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:34:56.736 
  command:

    streamlit run /opt/anaconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-07-15 19:34:56.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:34:56.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:34:56.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:34:56.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when run

DeltaGenerator()

## Step 2: Load Data

Cache your data for streamlit. In streamlit, any time a user interacts with a widget the whole script is rerun from top to bottom. By cacheing the data, we can speed up user interface

In [6]:
@st.cache_data
def load_data():
    df = pd.read_csv('nba_shots_optimized.csv')
    df = df.dropna(subset=['player.player_name', 'Season', 'player.team_name'])
    df['Season'] = df['Season'].astype(str).str.extract(r'(\d{4})')
    df['Season'] = df['Season'].astype(int)
    return df

try:
    df = load_data()
except FileNotFoundError:
    st.error("couldn't find nba_shots_optimized.csv. Is your data or path in the right spot?")
    st.stop()


2026-07-15 19:37:40.739 No runtime found, using MemoryCacheStorageManager
2026-07-15 19:37:40.741 No runtime found, using MemoryCacheStorageManager
2026-07-15 19:37:40.742 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:40.743 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:40.744 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:40.744 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:41.264 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:41.264 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 19:37:41.377 Thread 'Thread-6': missing ScriptRunContext! This warning can be ignored whe

## Setting Up the Sidebar Widgets

We're using Streamlit's st.sidebar to shove all our filters over to the left side of the screen. This keeps the main area clean and dedicated entirely to our shot charts.

The Player Dropdown:
st.sidebar.selectbox grabs every unique name from player.player_name, sorts them A-to-Z, and drops them into a searchable dropdown. Whichever name the user clicks gets saved to selected_player.

The Team Dropdown:
st.sidebar.selectbox grabs every unique name from player.team_name, sorts them A-to-Z, and drops them into a searchable dropdown. Whichever team the user clicks gets saved to selected_team.

The Season Slider:
st.sidebar.slider looks at the minimum and maximum years in our Season column. By passing a tuple of both (min_season, max_season) as the starting point, Streamlit automatically turns it into a range slider with two draggable handles.

The Shot Action Selector:
st.sidebar.multiselect gets a list of specific play types (like a Step Back Jump Shot or Running Dunk). We drop any empty values with .dropna() so the UI stays clean. This starts empty, meaning the user can select as many specific play types as they want—or none at all.

The Clutch Toggle:
A simple st.sidebar.checkbox that returns True or False. We also added a help hover-tooltip to explain exactly what "Clutch" means in our app.



In [41]:
st.sidebar.header("Dashboard Filters")



# Filter: Team Selector
all_teams = sorted(df['player.team_name'].unique().tolist())
selected_team = st.sidebar.multiselect("Select a Team", options=all_teams, default=[],help='Leave blank to include all teams.')

if selected_team:
    available_players_df = df[df['player.team_name'].isin(selected_team)]
else:
    available_players_df = df

# Filter: Player Selector
all_players = sorted(available_players_df['player.player_name'].unique())

selected_players = st.sidebar.multiselect(
    "Select Player(s)", 
    options=all_players, 
    default=[],
    help="Select one or multiple players. Leave empty to show all players."
)
# Filter: Season Range Slider
clean_seasons = df['Season'].dropna()
min_season = int(clean_seasons.min())
max_season = int(clean_seasons.max())
selected_seasons = st.sidebar.slider(
    "Select Season Range", 
    min_season, 
    max_season, 
    (min_season, max_season)
)

# Filter: Action Type
all_actions = sorted(df['player.action_type'].dropna().unique())
selected_actions = st.sidebar.multiselect(
    "Filter by Shot Action (Optional)", 
    all_actions, 
    default=[]
)

# Filter: Clutch Toggle
clutch_only = st.sidebar.checkbox(
    "Clutch Situations Only", 
    value=False,
    help="Filters for shots taken in the 4th Quarter or Overtime with 5 or fewer minutes remaining."
)

2026-07-15 21:05:21.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.044 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.045 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.090 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:05:21.091 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [58]:
filtered_df = df[
    (df['Season'].between(selected_seasons[0], selected_seasons[1])) &
    (df['player.shot_distance'] <=45) &
    (df['player.loc_y'] < 300)
].copy()

filtered_df['x_scaled'] = filtered_df['player.loc_x']*10
filtered_df['y_scaled'] = filtered_df['player.loc_y'] * 10
filtered_df['plot_y'] = (51.05 - filtered_df['player.loc_y']) * 10

filtered_df['plot_y'] = 500 - (filtered_df['y_scaled'] + 50)

if selected_players:
    filtered_df = filtered_df[filtered_df['player.player_name'].isin(selected_players)]

if 'selected_teams' in locals() and selected_teams:
    filtered_df = filtered_df[filtered_df['player.team_name'].isin(selected_teams)]
if selected_actions:
    filtered_df = filtered_df[filtered_df['player.action_type'].isin(selected_actions)]

if clutch_only:
    filtered_df = filtered_df[
        (filtered_df['player.period'] >= 4) & 
        (filtered_df['player.minutes_remaining'] <= 5)
    ]


In [54]:
total_shots = len(filtered_df)
if total_shots >0:
    made_shots = filtered_df['player.shot_made_numeric'].sum()
    fg_pct = (made_shots / total_shots) * 100
else:
    fg_pct = 0.0

#Header
players_label = ", ".join(selected_players) if selected_players else "No Players Selected"
st.markdown(f"### Current View: **{players_label}** | Total Shots: `{total_shots}` | FG%: `{fg_pct:.1f}%`")

2026-07-15 21:35:01.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:01.848 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:01.849 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [55]:
row1_col1, row1_col2 = st.columns(2)
row2_col1, row2_col2 = st.columns(2)

2026-07-15 21:35:05.735 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:05.736 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:05.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:05.737 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:05.738 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 21:35:05.738 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Above we set up our dashboard grid. By default, streamlit piles everything into one long column. st.columns() lets you break that into multiple columns. 

by using st.columns(2) we tell streamlit to split the horizontal width of the page into two equal side by side columns. to put stuff in the columns we can use the with statement

# Shot Locations

We'll use the plotly scatter functionality to build out our shots on court. We can map shot locations using the x_scaled and y_scaled to represent the x and y coordinates on the court, we create a precise representation of each shot attempt. We used a custom aligned half court background image layered behind the scatter data to ensure the alignment between the plotted coordinates and the actual court geometry. then we toggle the coloring to determine by player or by team

In [84]:
with row1_col1:
    st.markdown("#### Shot Charts")
    if not filtered_df.empty:
        color_column = 'player.team_name' if len(selected_team) > 1 else 'player.player_name'
        if len(selected_team) > 1:
            filtered_df = filtered_df[filtered_df['player.team_name'].isin(selected_team)]
            
        fig_map = px.scatter(
            filtered_df,
            x='x_scaled',
            y='y_scaled',
            symbol = 'player.shot_made_flag',
            symbol_map = {1: 'circle-open', 0: 'x'},
            color=color_column, 
            hover_data=['player.action_type', 'player.shot_distance', 'player.shot_made_flag'],
            opacity=0.8,
            title="Shot Coordinates"
        )

        court_image_path = "court image.png"
        if os.path.exists(court_image_path):
            with open(court_image_path, "rb") as img_file:
                encoded_string = base64.b64encode(img_file.read()).decode()
            court_data_uri = f"data:image/png;base64,{encoded_string}"

            fig_map.add_layout_image(
                dict(
                    source=court_data_uri,
                    xref="x",
                    yref="y",
                    x=-250,        
                    y=550,         
                    sizex=-500,     
                    sizey=575,     
                    sizing="stretch",
                    opacity=1.0,
                    layer="below"  
                )
            )
        else:
            st.warning("Could not find 'court.png' in your local directory. Please save the image there.")
        
        fig_map.update_xaxes(range=[-250, 250], showgrid=False, zeroline=False, visible=False)
        fig_map.update_yaxes(range=[-50, 550], scaleanchor='x',scaleratio=1, showgrid=False, zeroline=False, visible=False)
        
        fig_map.update_layout(
            width=500, 
            height=450, 
            legend_title="Player/Team",
            plot_bgcolor="rgba(0,0,0,0)",
            paper_bgcolor="rgba(0,0,0,0)",
            autosize=False,
            margin=dict(l=0, r=0, t=30, b=0),
            legend=dict(
                orientation='v',
                yanchor='top',
                xanchor='left',
                itemclick='toggleothers',
                itemsizing='constant',
                x=1.02,
                y=1
            )
        )
        st.plotly_chart(fig_map, width='content')
    else:
        st.info("No shot data available for this player combination.")        


2026-07-16 20:45:56.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:56.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:56.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:58.360 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:58.364 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:58.430 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:58.431 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-16 20:45:58.439 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Shot Breakdown

Here we use a stacked bar to get a visual distribution of the % of 2s and 3s taken over the selected seasons. We use this to determine efficiency of the shots. 

In [23]:
with row1_col2:
    st.markdown("#### 2-Pointers vs 3-Pointers")
    if not filtered_df.empty:
        type_by_season = (
            filtered_df.groupby(['Season', 'player.shot_type'], observed=False)
            .size()
            .reset_index(name='Shot Count')
        )
        
        fig_bar = px.bar(
            type_by_season,
            x='Season',
            y='Shot Count',
            color='player.shot_type',
            title="Yearly Shot Selection Breakdown",
            barmode='stack', 
            color_discrete_sequence=px.colors.qualitative.Set2,
            labels={'player.shot_type': 'Shot Type', 'Shot Count': 'Number of Attempts'}
        )
        
        fig_bar.update_layout(
            height=400,
            xaxis=dict(tickmode='linear'),
            legend_title="Shot Type"
        )
        
        st.plotly_chart(fig_bar, use_container_width=True)
    else:
        st.info("No shot data available.")

2026-07-15 20:15:33.615 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:15:33.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:15:33.617 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:15:33.652 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-07-15 20:15:33.653 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:15:33.654 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:15:33.654 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## Shot Distance Histogram

We can use a histogram to show the distance ranges teams and players are shooting from. We use this histogram to show the distribution of ranges where players are shooting from

In [24]:
with row2_col1:
    st.markdown("#### Shot Frequency by Distance")
    if not filtered_df.empty:
        filtered_df['distance_bin'] = pd.cut(
            filtered_df['player.shot_distance'], 
            bins=[-1, 5, 10, 15, 20, 25, 40], 
            labels=['0-5 ft', '6-10 ft', '11-15 ft', '16-20 ft', '21-25 ft', '26+ ft']
        )
        dist_grouped = filtered_df.groupby('distance_bin', observed=False).size().reset_index(name='Attempts')
        
        fig_dist = px.bar(
            dist_grouped,
            x='distance_bin',
            y='Attempts',
            labels={'distance_bin': 'Shot Distance (Feet)'},
            title="Attempts Volume by Distance Range"
        )
        fig_dist.update_layout(height=400)
        st.plotly_chart(fig_dist, use_container_width=True)
    else:
        st.info("No shot data available.")

2026-07-15 20:17:47.431 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:17:47.433 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:17:47.434 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
/var/folders/nc/rp01x1r53k12vk5f23w_z4zh0000gn/T/ipykernel_26262/263725117.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

2026-07-15 20:17:47.481 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-07

## FG% Trend

A line chart is the perfect medium to display a yearly trend for how efficient teams and players are.

In [25]:
with row2_col2:
    st.markdown("#### Efficiency Trend Across Seasons (2015-2020)")
    if not filtered_df.empty:
        trend_df = filtered_df.groupby('Season')['player.shot_made_numeric'].mean().reset_index()
        trend_df['Field Goal %'] = trend_df['player.shot_made_numeric'] * 100
        
        fig_trend = px.line(
            trend_df,
            x='Season',
            y='Field Goal %',
            markers=True,
            title="Yearly Shooting Accuracy Trend"
        )
        fig_trend.update_yaxes(range=[0, 100])
        fig_trend.update_layout(height=400, xaxis=dict(tickmode='linear'))
        st.plotly_chart(fig_trend, use_container_width=True)
    else:
        st.info("No progression data available.")

2026-07-15 20:21:01.634 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:21:01.636 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:21:01.637 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:21:01.677 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
2026-07-15 20:21:01.678 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:21:01.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-15 20:21:01.679 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
